# Energy-Aware Scheduling Benchmark

Compares the **probabilistic** (energy-aware) scheduler against the **round-robin** baseline across five workloads: three lightweight (addition, naive-fib, matrix-mul) and two compute-heavy variants (~5 s each) that give the scheduler more opportunity to demonstrate energy savings.

Each task records five timestamps used throughout this notebook to decompose end-to-end latency into phases:

| Timestamp | Phase boundary |
|---|---|
| `start_time` → `scheduler_time` | Scheduling overhead |
| `scheduler_time` → `proplet_arrive_time` | Queue / network transfer |
| `proplet_arrive_time` → `execution_time` | Wait for executor |
| `execution_time` → `finish_time` | Wasm execution |

# Setup & Helper functions

In [47]:
import os
# When the notebook lives in notebooks/, set cwd to the project root so that
# relative paths (build/, benchmark-results/, ...) resolve correctly.
if os.path.basename(os.getcwd()) == "notebooks":
    os.chdir("..")

import json
import requests
import scipy
import time
import dateutil
import matplotlib.pyplot as plt
import matplotlib
import concurrent
from concurrent.futures import ThreadPoolExecutor, wait, FIRST_COMPLETED, as_completed

In [48]:
SCHEDULERS = ["roundrobin", "probabilistic", "static"]
TASKS_URL = "http://localhost:7070/tasks"
THROUGHPUT_CLIENTS = [1, 2, 4, 8]
THROUGHPUT_DURATION = 30  # seconds
REP_COUNTS = [500]  # default; overridden per workload via "rep_counts"
MAX_TASK_ATTEMPTS = 5  # max attempts to create+upload or start a task before skipping it
WORKLOADS = {
    "addition": {
        "function_name": "add",
        "path": "addition.wasm",
        "inputs": [10, 20],
    },
    "naive-fib": {
        "function_name": "naive_fib",
        "path": "naive-fib.wasm",
        "inputs": [35],
    },
    "naive-fib-heavy": {
        "function_name": "naive_fib",
        "path": "naive-fib.wasm",
        "inputs": [40],
        "rep_counts": [50],
        "skip_warmup": True,
        "skip_throughput": True, # 30s window yields too few completions for TPS stats
    },
    "matrix-mul": {
        "function_name": "matrix_mul",
        "path": "matrix-mul.wasm",
        "inputs": [40],
    },
    "matrix-mul-heavy": {
        "function_name": "matrix_mul",
        "path": "matrix-mul.wasm",
        "inputs": [500],         # ~5s per task
        "rep_counts": [50],
        "skip_warmup": True,
        "skip_throughput": True,
    },
}

In [49]:
def create_task(task_name, task_inputs, scheduler = None) -> str:
    body = {"name": task_name, "inputs": task_inputs}
    if (scheduler):
        body["scheduler"] = scheduler
    resp = requests.post(
        TASKS_URL,
        headers={"Content-Type": "application/json"},
        data=json.dumps(body),
    )
    resp.raise_for_status()
    task_id = resp.json()["id"]
    return task_id

def upload_wasm(task_id, task_location):
    with open(task_location, "rb") as f:
        upload = requests.put(
            f"{TASKS_URL}/{task_id}/upload",
            files={"file": f},
        )
        upload.raise_for_status()

def prepare_tasks(task_count, task_name, task_location, task_inputs, scheduler = None):
    task_ids = []
    for i in range(task_count):
        print(f"Creating task {i+1}/{task_count}...")
        for attempt in range(1, MAX_TASK_ATTEMPTS + 1):
            try:
                task_id = create_task(task_name, task_inputs, scheduler)
                upload_wasm(task_id, task_location)
                task_ids.append(task_id)
                break
            except Exception as e:
                if attempt == MAX_TASK_ATTEMPTS:
                    print(f"  Task {i+1} failed after {MAX_TASK_ATTEMPTS} attempts, skipping: {e}")
                else:
                    print(f"  Task {i+1} attempt {attempt} failed, retrying: {e}")
    return task_ids

In [50]:
def start_task(task_id):
    resp = requests.post(f"{TASKS_URL}/{task_id}/start")
    resp.raise_for_status()

def start_tasks(task_ids):
    started = []
    for task_id in task_ids:
        for attempt in range(1, MAX_TASK_ATTEMPTS + 1):
            try:
                start_task(task_id)
                started.append(task_id)
                break
            except Exception as e:
                if attempt == MAX_TASK_ATTEMPTS:
                    print(f"  Failed to start task {task_id} after {MAX_TASK_ATTEMPTS} attempts, skipping: {e}")
                else:
                    print(f"  Start attempt {attempt} for task {task_id} failed, retrying: {e}")
    return started

In [51]:
def get_task_result(task_id):
    while True:
        resp = requests.get(f"{TASKS_URL}/{task_id}")
        resp.raise_for_status()
        data = resp.json()
        if "results" in data:
            return data
            break
        time.sleep(0.005)

def get_tasks_results(task_ids):
    results = []
    for task_id in task_ids:
        data = get_task_result(task_id)
        results.append(data)
    return results

In [52]:
def execute_benchmark(function_name, task_location, task_inputs, task_count = 1, scheduler = None):
    print(f"Starting test execution for task: {function_name}")

    task_ids = prepare_tasks(task_count, function_name, task_location, task_inputs, scheduler)
    print(f"Prepared {len(task_ids)}/{task_count} tasks successfully")

    start_execution = time.perf_counter_ns()

    # Start all tasks before polling any result, so they run concurrently on the proplets.
    started_ids = start_tasks(task_ids)
    print(f"Started {len(started_ids)}/{len(task_ids)} tasks successfully")

    results = get_tasks_results(started_ids)

    duration = (time.perf_counter_ns() - start_execution) * 1000 # ms
    print(f"Retrieved all task results successfully, start to finish took {duration} ms")

    return results, duration

In [53]:
def _submit_task_start(workload, input_data, scheduler, task_location):
    task_id = create_task(workload, input_data, scheduler)
    upload_wasm(task_id, task_location)
    start_task(task_id)
    return task_id

def _to_unix_seconds(ts):
    """Return a Unix timestamp in seconds, or None if the value is absent/unset.

    Go serialises time.Time zero values as "0001-01-01T00:00:00Z". These are
    fields that the runtime never populated (e.g. proplet_arrive_time before a
    task reaches a proplet). Treating them as None lets all callers skip the
    phase safely via a simple truthiness check.
    """
    if ts is None:
        return None
    if isinstance(ts, (int, float)):
        return float(ts) if ts > 0 else None  # 0 = Unix epoch, also not a real task time
    parsed = dateutil.parser.isoparse(ts)
    if parsed.year < 2:   # Go zero time is year 0001
        return None
    return parsed.timestamp()

In [54]:
def throughput_benchmark(scheduler, workload, input_data, task_location, concurrency, duration_sec=30):
    submitted_task_ids = []
    submit_start = time.time()
    submit_deadline = submit_start + duration_sec

    with ThreadPoolExecutor(max_workers=concurrency) as executor:
        in_flight = set()

        while time.time() < submit_deadline:
            while len(in_flight) < concurrency and time.time() < submit_deadline:
                fut = executor.submit(_submit_task_start, workload, input_data, scheduler, task_location)
                in_flight.add(fut)

            done, _ = wait(in_flight, timeout=0.0, return_when=FIRST_COMPLETED)
            for fut in done:
                in_flight.remove(fut)
                try:
                    task_id = fut.result()
                    submitted_task_ids.append(task_id)
                except Exception:
                    pass

        for fut in as_completed(in_flight):
            try:
                task_id = fut.result()
                submitted_task_ids.append(task_id)
            except Exception:
                pass

    # Record the cutoff immediately after the submission window closes. Tasks that finish
    # after this point are excluded from TPS — they benefited from extra time beyond the window.
    cutoff_unix = time.time()

    finished_before_cutoff = []
    for task_id in submitted_task_ids:
        try:
            data = get_task_result(task_id)
            finish_unix = _to_unix_seconds(data.get("finish_time"))
            if finish_unix is not None and finish_unix <= cutoff_unix:
                finished_before_cutoff.append(data)
        except Exception:
            pass

    throughput = len(finished_before_cutoff) / duration_sec

    return {
        "submitted": len(submitted_task_ids),
        "completed_before_cutoff": len(finished_before_cutoff),
        "throughput_tps": throughput,
        "cutoff_unix": cutoff_unix,
        "results_before_cutoff": finished_before_cutoff,
    }

# Benchmark Execution

## Warmup

Runs tasks for 180 s before measuring. This brings the system to a thermal steady state so that measured energy and latency values reflect stable operating conditions rather than cold-start behaviour. Heavy workloads are excluded to stay within the time budget.

In [55]:
WARMUP_BATCH_SIZE = 50
WARMUP_DURATION = 180  # seconds — enough time for thermal plateau

print("Warming up system...")
warmup_deadline = time.time() + WARMUP_DURATION
iteration = 0
while time.time() < warmup_deadline:
    for task_name, task_info in WORKLOADS.items():
        if task_info.get("skip_warmup"):
            continue
        execute_benchmark(
            function_name=task_info["function_name"],
            task_count=WARMUP_BATCH_SIZE,
            task_location=f"build/{task_info['path']}",
            task_inputs=task_info["inputs"],
            scheduler=SCHEDULERS[0],
        )
        if time.time() >= warmup_deadline:
            break
    iteration += 1
    print(f"  Warmup iteration {iteration} done, {max(0, warmup_deadline - time.time()):.0f}s remaining")
print("Warmup complete.")

Warming up system...
Starting test execution for task: add
Creating task 1/50...
Creating task 2/50...
Creating task 3/50...
Creating task 4/50...
Creating task 5/50...
Creating task 6/50...
Creating task 7/50...
Creating task 8/50...
Creating task 9/50...
Creating task 10/50...
Creating task 11/50...
Creating task 12/50...
Creating task 13/50...
Creating task 14/50...
Creating task 15/50...
Creating task 16/50...
Creating task 17/50...
Creating task 18/50...
Creating task 19/50...
Creating task 20/50...
Creating task 21/50...
Creating task 22/50...
Creating task 23/50...
Creating task 24/50...
Creating task 25/50...
Creating task 26/50...
Creating task 27/50...
Creating task 28/50...
Creating task 29/50...
Creating task 30/50...
Creating task 31/50...
Creating task 32/50...
Creating task 33/50...
Creating task 34/50...
Creating task 35/50...
Creating task 36/50...
Creating task 37/50...
Creating task 38/50...
Creating task 39/50...
Creating task 40/50...
Creating task 41/50...
Creatin

## Latency

Submits `rep_count` tasks per scheduler/workload combination (500 for lightweight workloads, 50 for heavy ones), starts them all, then polls for results. The raw task records — including all five timestamps and `energy_consumed` — are stored in `results` and analysed in the Results section below.

In [56]:
results = {}

for scheduler in SCHEDULERS:
    for task_name, task_info in WORKLOADS.items():
        for rep_count in task_info.get("rep_counts", REP_COUNTS):
            print(f"Running benchmark for scheduler: {scheduler}, task: {task_name}, rep count: {rep_count}")
            task_results, duration = execute_benchmark(
                function_name=task_info["function_name"],
                task_count=rep_count,
                task_location=f"build/{task_info['path']}",
                task_inputs=task_info["inputs"],
                scheduler=scheduler,
            )
            results[f"{scheduler}_{task_name}_{rep_count}"] = {
                "results": task_results,
                "duration": duration,
            }

Running benchmark for scheduler: roundrobin, task: addition, rep count: 500
Starting test execution for task: add
Creating task 1/500...
Creating task 2/500...
Creating task 3/500...
Creating task 4/500...
Creating task 5/500...
Creating task 6/500...
Creating task 7/500...
Creating task 8/500...
Creating task 9/500...
Creating task 10/500...
Creating task 11/500...
Creating task 12/500...
Creating task 13/500...
Creating task 14/500...
Creating task 15/500...
Creating task 16/500...
Creating task 17/500...
Creating task 18/500...
Creating task 19/500...
Creating task 20/500...
Creating task 21/500...
Creating task 22/500...
Creating task 23/500...
Creating task 24/500...
Creating task 25/500...
Creating task 26/500...
Creating task 27/500...
Creating task 28/500...
Creating task 29/500...
Creating task 30/500...
Creating task 31/500...
Creating task 32/500...
Creating task 33/500...
Creating task 34/500...
Creating task 35/500...
Creating task 36/500...
Creating task 37/500...
Creatin

## Throughput

Submits tasks continuously at each concurrency level for `THROUGHPUT_DURATION` seconds. TPS is computed as tasks completed before the submission window closed divided by the window length. Heavy workloads are excluded — at ~5 s per task, a 30 s window yields too few completions for a meaningful TPS figure.

In [57]:
throughput_benchmark_results = {}

for scheduler in SCHEDULERS:
    print(f"Running throughput benchmark for scheduler: {scheduler}")
    for task_name, task_info in WORKLOADS.items():
        if task_info.get("skip_throughput"):
            continue
        print(f"Running throughput benchmark for task: {task_name}")
        for concurrency in THROUGHPUT_CLIENTS:
            result = throughput_benchmark(
                scheduler=scheduler,
                workload=task_info["function_name"],
                input_data=task_info["inputs"],
                task_location=f"build/{task_info['path']}",
                concurrency=concurrency,
                duration_sec=THROUGHPUT_DURATION,
            )
            throughput_benchmark_results[f"{scheduler}_{task_name}_{concurrency}"] = {
                "scheduler": scheduler,
                "task": task_name,
                "concurrency": concurrency,
                "submitted": result["submitted"],
                "completed_before_cutoff": result["completed_before_cutoff"],
                "throughput_tps": result["throughput_tps"],
            }

Running throughput benchmark for scheduler: roundrobin
Running throughput benchmark for task: addition
Running throughput benchmark for task: naive-fib
Running throughput benchmark for task: matrix-mul
Running throughput benchmark for scheduler: probabilistic
Running throughput benchmark for task: addition
Running throughput benchmark for task: naive-fib
Running throughput benchmark for task: matrix-mul
Running throughput benchmark for scheduler: static
Running throughput benchmark for task: addition
Running throughput benchmark for task: naive-fib
Running throughput benchmark for task: matrix-mul


# Results

Run these cells after the benchmark execution above has completed. Charts are saved to `benchmark-results/`; numerical statistics are printed inline and exported to CSV at the end.

## Combined Figures

One figure per metric, each with one subplot per workload (5 subplots in a 2×3 grid), labelled (a)–(e). Saved to `benchmark-results/`:
- `load_balance_combined.png` — task fraction per proplet, grouped by scheduler
- `energy_combined.png` — per-task energy distribution (boxplot per scheduler; mJ for addition/matrix-mul, J for the rest)
- `total_energy_combined.png` — total energy consumed per scheduler (bar chart; same units as above)
- `latency_combined.png` — end-to-end latency distribution (boxplot per scheduler)
- `latency_phases_combined.png` — mean latency phase breakdown (stacked bar per scheduler)
- `throughput_combined.png` — throughput scaling (TPS vs concurrency, lightweight workloads only; labelled (a)–(c))

In [72]:
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import numpy as np

os.makedirs("benchmark-results", exist_ok=True)

workload_list = list(WORKLOADS.keys())
n_wl = len(workload_list)

# Per-workload energy unit and conversion factor from stored mJ
ENERGY_UNIT = {
    "addition":         ("mJ", 1.0),
    "naive-fib":        ("J",  1e-3),
    "naive-fib-heavy":  ("J",  1e-3),
    "matrix-mul":       ("mJ", 1.0),
    "matrix-mul-heavy": ("J",  1e-3),
}

def make_combined_fig(n_subplots, figsize=(16, 9)):
    ncols = 3
    nrows = (n_subplots + ncols - 1) // ncols
    fig, axes = plt.subplots(nrows, ncols, figsize=figsize)
    axes_flat = axes.flatten() if hasattr(axes, "flatten") else [axes]
    for i in range(n_subplots, len(axes_flat)):
        axes_flat[i].set_visible(False)
    return fig, axes_flat

def add_subplot_label(ax, idx):
    ax.text(0.02, 0.97, f"({chr(97 + idx)})", transform=ax.transAxes,
            ha="left", va="top", fontsize=9, fontweight="bold",
            bbox=dict(boxstyle="round,pad=0.2", facecolor="white", edgecolor="none", alpha=0.7))

# ── Load balance ───────────────────────────────────────────────────────────────
fig, axes = make_combined_fig(n_wl)

for idx, task_name in enumerate(workload_list):
    ax = axes[idx]
    task_info = WORKLOADS[task_name]
    rep_count = task_info.get("rep_counts", REP_COUNTS)[0]

    all_pids_set = set()
    pid_counts_per_sched = {}
    for scheduler in SCHEDULERS:
        value = results.get(f"{scheduler}_{task_name}_{rep_count}")
        counts = {}
        if value:
            for t in value["results"]:
                pid = t["proplet_id"]
                counts[pid] = counts.get(pid, 0) + 1
                all_pids_set.add(pid)
        pid_counts_per_sched[scheduler] = counts

    all_pids = sorted(all_pids_set)
    label_map = {pid: f"P{i+1}" for i, pid in enumerate(all_pids)}
    proplet_colors = [f"C{j}" for j in range(len(all_pids))]

    x = np.arange(len(SCHEDULERS))
    width = 0.8 / max(len(all_pids), 1)

    for j, pid in enumerate(all_pids):
        fracs = [
            pid_counts_per_sched[s].get(pid, 0) / max(sum(pid_counts_per_sched[s].values()), 1)
            for s in SCHEDULERS
        ]
        bars = ax.bar(x + j * width, fracs, width, label=label_map[pid], color=proplet_colors[j])
        for bar, frac in zip(bars, fracs):
            ax.text(bar.get_x() + bar.get_width() / 2, frac + 0.01, f"{frac:.2f}",
                    ha="center", va="bottom", fontsize=6)

    ax.set_xticks(x + width * (len(all_pids) - 1) / 2)
    ax.set_xticklabels(SCHEDULERS, rotation=15, ha="right", fontsize=8)
    ax.set_ylabel("Fraction of tasks", fontsize=9)
    ax.legend(title="Proplet", fontsize=7, title_fontsize=8)
    ax.yaxis.grid(True, zorder=0)
    ax.set_ylim(0, 1.15)
    add_subplot_label(ax, idx)

plt.tight_layout()
plt.savefig("benchmark-results/load_balance_combined.png", dpi=100, bbox_inches="tight")
plt.close(fig)
print("Saved load_balance_combined.png")

# ── Per-task energy (boxplots) ─────────────────────────────────────────────────
fig, axes = make_combined_fig(n_wl)

for idx, task_name in enumerate(workload_list):
    ax = axes[idx]
    task_info = WORKLOADS[task_name]
    rep_count = task_info.get("rep_counts", REP_COUNTS)[0]
    unit, scale = ENERGY_UNIT.get(task_name, ("mJ", 1.0))

    box_data = []
    for scheduler in SCHEDULERS:
        value = results.get(f"{scheduler}_{task_name}_{rep_count}")
        vals = [t["energy_consumed"] * scale for t in value["results"]
                if t.get("energy_consumed") is not None] if value else []
        box_data.append(vals)

    ax.boxplot(box_data)
    ax.set_xticks(range(1, len(SCHEDULERS) + 1))
    ax.set_xticklabels(SCHEDULERS, rotation=15, ha="right", fontsize=8)
    ax.set_ylabel(f"Energy consumed ({unit})", fontsize=9)
    ax.yaxis.grid(True)
    add_subplot_label(ax, idx)

plt.tight_layout()
plt.savefig("benchmark-results/energy_combined.png", dpi=100, bbox_inches="tight")
plt.close(fig)
print("Saved energy_combined.png")

# ── Total energy (bar chart) ───────────────────────────────────────────────────
fig, axes = make_combined_fig(n_wl)

for idx, task_name in enumerate(workload_list):
    ax = axes[idx]
    task_info = WORKLOADS[task_name]
    rep_count = task_info.get("rep_counts", REP_COUNTS)[0]
    unit, scale = ENERGY_UNIT.get(task_name, ("mJ", 1.0))

    total_energy = []
    for scheduler in SCHEDULERS:
        value = results.get(f"{scheduler}_{task_name}_{rep_count}")
        total = sum(t["energy_consumed"] for t in value["results"]
                    if t.get("energy_consumed") is not None) * scale if value else 0
        total_energy.append(total)

    bars = ax.bar(SCHEDULERS, total_energy)
    for bar, val in zip(bars, total_energy):
        ax.annotate(f"{val:.2f}", (bar.get_x() + bar.get_width() / 2, bar.get_height()),
                    ha="center", va="bottom", fontsize=8, xytext=(0, 3), textcoords="offset points")
    ax.set_xticks(range(len(SCHEDULERS)))
    ax.set_xticklabels(SCHEDULERS, rotation=15, ha="right", fontsize=8)
    ax.set_ylabel(f"Total energy ({unit})", fontsize=9)
    ax.yaxis.grid(True)
    add_subplot_label(ax, idx)

plt.tight_layout()
plt.savefig("benchmark-results/total_energy_combined.png", dpi=100, bbox_inches="tight")
plt.close(fig)
print("Saved total_energy_combined.png")

# ── Latency e2e (boxplots) ─────────────────────────────────────────────────────
fig, axes = make_combined_fig(n_wl)

for idx, task_name in enumerate(workload_list):
    ax = axes[idx]
    task_info = WORKLOADS[task_name]
    rep_count = task_info.get("rep_counts", REP_COUNTS)[0]

    box_data = []
    for scheduler in SCHEDULERS:
        value = results.get(f"{scheduler}_{task_name}_{rep_count}")
        vals = []
        if value:
            for t in value["results"]:
                start = _to_unix_seconds(t.get("start_time"))
                finish = _to_unix_seconds(t.get("finish_time"))
                if start and finish:
                    vals.append((finish - start) * 1000)
        box_data.append(vals)

    ax.boxplot(box_data)
    ax.set_xticks(range(1, len(SCHEDULERS) + 1))
    ax.set_xticklabels(SCHEDULERS, rotation=15, ha="right", fontsize=8)
    ax.set_ylabel("Latency (ms)", fontsize=9)
    ax.yaxis.grid(True)
    add_subplot_label(ax, idx)

plt.tight_layout()
plt.savefig("benchmark-results/latency_combined.png", dpi=100, bbox_inches="tight")
plt.close(fig)
print("Saved latency_combined.png")

Saved load_balance_combined.png
Saved energy_combined.png
Saved total_energy_combined.png
Saved latency_combined.png


## Energy Efficiency

Per-task energy statistics (mean, std, percentiles, 95 % CI) for each scheduler/workload combination. Statistical significance is tested with Mann-Whitney U (non-parametric); practical significance with Cliff's delta (|d| < 0.147 negligible, < 0.33 small, < 0.474 medium, ≥ 0.474 large).

In [73]:
from scipy import stats as scipy_stats
import numpy as np

print("=== Energy Consumption Statistics ===\n")
for task_name, task_info in WORKLOADS.items():
    for rep_count in task_info.get("rep_counts", REP_COUNTS):
        print(f"Task: {task_name}, n={rep_count}")
        energy_by_sched = {}
        for scheduler in SCHEDULERS:
            value = results.get(f"{scheduler}_{task_name}_{rep_count}")
            if value:
                energy_by_sched[scheduler] = np.array(
                    [t["energy_consumed"] for t in value["results"] if t.get("energy_consumed") is not None],
                    dtype=float,
                )

        for scheduler, arr in energy_by_sched.items():
            ci = scipy_stats.t.interval(0.95, len(arr) - 1, loc=arr.mean(), scale=scipy_stats.sem(arr))
            print(f"  {scheduler}: mean={arr.mean():.3f}, std={arr.std(ddof=1):.3f}, "
                  f"p50={np.median(arr):.3f}, p95={np.percentile(arr, 95):.3f}, p99={np.percentile(arr, 99):.3f}, "
                  f"total={arr.sum():.1f}, CI95=[{ci[0]:.3f}, {ci[1]:.3f}], n={len(arr)}")

        baseline = energy_by_sched.get("roundrobin")
        if baseline is not None:
            for sched_b in SCHEDULERS:
                if sched_b == "roundrobin" or sched_b not in energy_by_sched:
                    continue
                b = energy_by_sched[sched_b]
                _, p = scipy_stats.mannwhitneyu(baseline, b, alternative="two-sided")
                comparisons = np.sign(baseline[:, None] - b[None, :])
                cliffs_d = float(comparisons.mean())
                mag = ("negligible" if abs(cliffs_d) < 0.147 else
                       "small"      if abs(cliffs_d) < 0.330 else
                       "medium"     if abs(cliffs_d) < 0.474 else "large")
                sig = "*" if p < 0.05 else ""
                rr_total = baseline.sum()
                savings_pct = (rr_total - b.sum()) / rr_total * 100 if rr_total > 0 else float("nan")
                print(f"  roundrobin vs {sched_b}: Mann-Whitney U p={p:.4f}{sig}  "
                      f"Cliff's d={cliffs_d:+.3f} ({mag})  savings={savings_pct:.2f}%")
        print()

=== Energy Consumption Statistics ===

Task: addition, n=500
  roundrobin: mean=2.205, std=0.893, p50=1.989, p95=3.518, p99=4.276, total=1102.6, CI95=[2.127, 2.284], n=500
  probabilistic: mean=2.076, std=0.604, p50=1.926, p95=3.220, p99=3.557, total=1037.9, CI95=[2.023, 2.129], n=500
  static: mean=2.256, std=0.719, p50=2.055, p95=3.575, p99=3.940, total=1127.8, CI95=[2.192, 2.319], n=500
  roundrobin vs probabilistic: Mann-Whitney U p=0.0778  Cliff's d=+0.064 (negligible)  savings=5.87%
  roundrobin vs static: Mann-Whitney U p=0.0673  Cliff's d=-0.067 (negligible)  savings=-2.28%

Task: naive-fib, n=500
  roundrobin: mean=690.014, std=200.682, p50=649.096, p95=1059.126, p99=1192.093, total=345007.1, CI95=[672.381, 707.647], n=500
  probabilistic: mean=695.751, std=197.647, p50=668.353, p95=1100.645, p99=1186.124, total=347875.6, CI95=[678.385, 713.118], n=500
  static: mean=864.405, std=249.661, p50=840.696, p95=1272.742, p99=1330.318, total=432202.4, CI95=[842.468, 886.341], n=500
 

## Latency

Decomposes each task's end-to-end latency into the four phases defined in the introduction, then reports descriptive statistics and Mann-Whitney U + Cliff's delta tests for the three most relevant phases: e2e, scheduling overhead, and Wasm execution. A combined stacked bar chart follows (`latency_phases_combined.png`).

In [74]:
import numpy as np
from scipy import stats as scipy_stats

def _phase_stats(arr):
    if len(arr) < 2:
        return {}
    a = np.array(arr, dtype=float)
    ci = scipy_stats.t.interval(0.95, len(a) - 1, loc=a.mean(), scale=scipy_stats.sem(a))
    return {
        "n": len(a),
        "mean":     float(a.mean()),
        "std":      float(a.std(ddof=1)),
        "median":   float(np.median(a)),
        "p95":      float(np.percentile(a, 95)),
        "p99":      float(np.percentile(a, 99)),
        "iqr":      float(np.percentile(a, 75) - np.percentile(a, 25)),
        "ci95_low":  float(ci[0]),
        "ci95_high": float(ci[1]),
    }

latency_stats = {}

print("=== Latency Phase Breakdown (ms) ===\n")
for task_name, task_info in WORKLOADS.items():
    for rep_count in task_info.get("rep_counts", REP_COUNTS):
        print(f"Task: {task_name}, n={rep_count}")
        phase_data_by_sched = {}

        for scheduler in SCHEDULERS:
            value = results.get(f"{scheduler}_{task_name}_{rep_count}")
            if not value:
                continue
            phases = {
                "e2e":                 [],
                "scheduling_overhead": [],
                "queue_transfer":      [],
                "wait_for_exec":       [],
                "wasm_execution":      [],
            }
            for t in value["results"]:
                start     = _to_unix_seconds(t.get("start_time"))
                sched     = _to_unix_seconds(t.get("scheduler_time"))
                arrive    = _to_unix_seconds(t.get("proplet_arrive_time"))
                exec_s    = _to_unix_seconds(t.get("execution_time"))
                finish    = _to_unix_seconds(t.get("finish_time"))
                if start and finish:
                    phases["e2e"].append((finish - start) * 1000)
                if start and sched:
                    phases["scheduling_overhead"].append((sched - start) * 1000)
                if sched and arrive:
                    phases["queue_transfer"].append((arrive - sched) * 1000)
                if arrive and exec_s:
                    phases["wait_for_exec"].append((exec_s - arrive) * 1000)
                if exec_s and finish:
                    phases["wasm_execution"].append((finish - exec_s) * 1000)

            phase_data_by_sched[scheduler] = phases
            latency_stats[f"{scheduler}_{task_name}_{rep_count}"] = {
                k: _phase_stats(v) for k, v in phases.items()
            }
            print(f"  {scheduler}:")
            for phase_name, vals in phases.items():
                s = _phase_stats(vals)
                if s:
                    print(f"    {phase_name:25s}: mean={s['mean']:8.3f}  std={s['std']:8.3f}  "
                          f"p50={s['median']:8.3f}  p95={s['p95']:8.3f}  p99={s['p99']:8.3f}  "
                          f"CI95=[{s['ci95_low']:.3f}, {s['ci95_high']:.3f}]")

        baseline_phases = phase_data_by_sched.get("roundrobin")
        if baseline_phases is not None:
            for sched_b in SCHEDULERS:
                if sched_b == "roundrobin" or sched_b not in phase_data_by_sched:
                    continue
                print(f"  Statistical tests (roundrobin vs {sched_b}):")
                for phase_name in ["e2e", "scheduling_overhead", "wasm_execution"]:
                    a_vals = np.array(baseline_phases[phase_name], dtype=float)
                    b_vals = np.array(phase_data_by_sched[sched_b][phase_name], dtype=float)
                    if len(a_vals) < 2 or len(b_vals) < 2:
                        continue
                    _, p = scipy_stats.mannwhitneyu(a_vals, b_vals, alternative="two-sided")
                    comparisons = np.sign(a_vals[:, None] - b_vals[None, :])
                    cliffs_d = float(comparisons.mean())
                    mag = ("negligible" if abs(cliffs_d) < 0.147 else
                           "small"      if abs(cliffs_d) < 0.330 else
                           "medium"     if abs(cliffs_d) < 0.474 else "large")
                    sig = "*" if p < 0.05 else ""
                    print(f"    {phase_name:25s}: p={p:.4f}{sig:1s}  Cliff's d={cliffs_d:+.3f} ({mag})")
        print()

=== Latency Phase Breakdown (ms) ===

Task: addition, n=500
  roundrobin:
    e2e                      : mean= 304.661  std=  84.668  p50= 278.831  p95= 470.583  p99= 524.746  CI95=[297.222, 312.101]
    scheduling_overhead      : mean=   0.017  std=   0.019  p50=   0.013  p95=   0.029  p99=   0.049  CI95=[0.015, 0.018]
    queue_transfer           : mean=  58.123  std=  59.578  p50=  29.985  p95= 208.318  p99= 254.529  CI95=[52.888, 63.358]
  probabilistic:
    e2e                      : mean= 316.542  std=  80.390  p50= 299.723  p95= 457.442  p99= 493.394  CI95=[309.479, 323.606]
    scheduling_overhead      : mean=   0.014  std=   0.013  p50=   0.013  p95=   0.022  p99=   0.032  CI95=[0.013, 0.015]
    queue_transfer           : mean=  57.488  std=  58.780  p50=  29.538  p95= 202.168  p99= 235.781  CI95=[52.323, 62.653]
  static:
    e2e                      : mean= 479.065  std= 272.267  p50= 417.156  p95=1185.407  p99=1358.823  CI95=[455.142, 502.988]
    scheduling_overhead      

## Load Balance

Measures how evenly tasks are distributed across proplets. CV (coefficient of variation) and Gini coefficient both equal 0 for perfect equality; higher values indicate skew. This quantifies whether the probabilistic scheduler's energy routing produces unequal load distribution as a side effect.

In [75]:
print("=== Load Balance Quality ===\n")
for task_name, task_info in WORKLOADS.items():
    for rep_count in task_info.get("rep_counts", REP_COUNTS):
        print(f"Task: {task_name}, n={rep_count}")
        for scheduler in SCHEDULERS:
            value = results.get(f"{scheduler}_{task_name}_{rep_count}")
            if not value:
                continue
            counts = {}
            for t in value["results"]:
                pid = t.get("proplet_id", "unknown")
                counts[pid] = counts.get(pid, 0) + 1

            count_arr = np.array(list(counts.values()), dtype=float)
            n = len(count_arr)
            cv = count_arr.std(ddof=1) / count_arr.mean() if count_arr.mean() > 0 else float("nan")
            sorted_c = np.sort(count_arr)
            gini = (2.0 * np.dot(np.arange(1, n + 1), sorted_c) - (n + 1) * sorted_c.sum()) / (n * sorted_c.sum()) if sorted_c.sum() > 0 else float("nan")
            dist = {f"P{i+1}": int(v) for i, v in enumerate(count_arr)}
            total = int(count_arr.sum())
            fractions = {k: f"{v/total:.3f}" for k, v in dist.items()}
            print(f"  {scheduler}: proplets={n}, tasks={dist}, fractions={fractions}, CV={cv:.4f}, Gini={gini:.4f}")
        print()

=== Load Balance Quality ===

Task: addition, n=500
  roundrobin: proplets=3, tasks={'P1': 167, 'P2': 167, 'P3': 166}, fractions={'P1': '0.334', 'P2': '0.334', 'P3': '0.332'}, CV=0.0035, Gini=0.0013
  probabilistic: proplets=3, tasks={'P1': 123, 'P2': 211, 'P3': 166}, fractions={'P1': '0.246', 'P2': '0.422', 'P3': '0.332'}, CV=0.2640, Gini=0.1173
  static: proplets=3, tasks={'P1': 169, 'P2': 111, 'P3': 220}, fractions={'P1': '0.338', 'P2': '0.222', 'P3': '0.440'}, CV=0.3272, Gini=0.1453

Task: naive-fib, n=500
  roundrobin: proplets=3, tasks={'P1': 167, 'P2': 167, 'P3': 166}, fractions={'P1': '0.334', 'P2': '0.334', 'P3': '0.332'}, CV=0.0035, Gini=0.0013
  probabilistic: proplets=3, tasks={'P1': 209, 'P2': 177, 'P3': 114}, fractions={'P1': '0.418', 'P2': '0.354', 'P3': '0.228'}, CV=0.2900, Gini=0.1267
  static: proplets=3, tasks={'P1': 148, 'P2': 146, 'P3': 206}, fractions={'P1': '0.296', 'P2': '0.292', 'P3': '0.412'}, CV=0.2045, Gini=0.0800

Task: naive-fib-heavy, n=50
  roundrobin: p

In [76]:
phase_order  = ["scheduling_overhead", "queue_transfer", "wait_for_exec", "wasm_execution"]
phase_colors = ["#4C72B0", "#DD8452", "#55A868", "#C44E52"]
phase_labels = ["Scheduling overhead", "Queue / transfer", "Wait for executor", "Wasm execution"]

# ── Latency phase breakdown (stacked bar) ──────────────────────────────────────
fig, axes = make_combined_fig(n_wl)

for idx, task_name in enumerate(workload_list):
    ax = axes[idx]
    task_info = WORKLOADS[task_name]
    rep_count = task_info.get("rep_counts", REP_COUNTS)[0]

    x = np.arange(len(SCHEDULERS))
    width = 0.5
    bottoms = np.zeros(len(SCHEDULERS))

    for color, phase, label in zip(phase_colors, phase_order, phase_labels):
        means = [
            latency_stats.get(f"{s}_{task_name}_{rep_count}", {}).get(phase, {}).get("mean", 0)
            for s in SCHEDULERS
        ]
        ax.bar(x, means, width, bottom=bottoms, label=label, color=color)
        bottoms += np.array(means)

    ax.set_xticks(x)
    ax.set_xticklabels(SCHEDULERS, rotation=15, ha="right", fontsize=8)
    ax.set_ylabel("Mean latency (ms)", fontsize=9)
    ax.yaxis.grid(True, zorder=0)
    add_subplot_label(ax, idx)

handles = [plt.Rectangle((0, 0), 1, 1, color=c) for c in phase_colors]
axes[n_wl - 1].legend(handles, phase_labels, loc="upper right", fontsize=7)

plt.tight_layout()
plt.savefig("benchmark-results/latency_phases_combined.png", dpi=100, bbox_inches="tight")
plt.close(fig)
print("Saved latency_phases_combined.png")

Saved latency_phases_combined.png


## Throughput

TPS at each concurrency level for the lightweight workloads. Saved to `benchmark-results/throughput_combined.png`.

In [77]:
print("=== Throughput Statistics (tasks/sec) ===\n")
throughput_workloads = {k: v for k, v in WORKLOADS.items() if not v.get("skip_throughput")}
n_tp = len(throughput_workloads)

fig, axes = plt.subplots(1, n_tp, figsize=(5 * n_tp, 5))
if n_tp == 1:
    axes = [axes]

for idx, (ax, (task_name, task_info)) in enumerate(zip(axes, throughput_workloads.items())):
    print(f"Task: {task_name}")
    for scheduler in SCHEDULERS:
        throughputs = []
        for concurrency in THROUGHPUT_CLIENTS:
            result = throughput_benchmark_results.get(f"{scheduler}_{task_name}_{concurrency}")
            if result:
                tps = result["throughput_tps"]
                throughputs.append(tps)
                print(f"  {scheduler}, concurrency={concurrency}: "
                      f"tps={tps:.2f}, submitted={result['submitted']}, "
                      f"completed={result['completed_before_cutoff']}")
            else:
                throughputs.append(0)
        ax.plot(THROUGHPUT_CLIENTS, throughputs, marker="o", label=scheduler)
    print()
    ax.set_xlabel("Concurrency (parallel clients)", fontsize=9)
    ax.set_ylabel("Throughput (tasks/sec)", fontsize=9)
    ax.legend(fontsize=8)
    ax.grid(True)
    add_subplot_label(ax, idx)

plt.tight_layout()
plt.savefig("benchmark-results/throughput_combined.png", dpi=100, bbox_inches="tight")
plt.close(fig)
print("Saved throughput_combined.png")

=== Throughput Statistics (tasks/sec) ===

Task: addition
  roundrobin, concurrency=1: tps=5.77, submitted=174, completed=173
  roundrobin, concurrency=2: tps=11.70, submitted=355, completed=351
  roundrobin, concurrency=4: tps=9.83, submitted=358, completed=295
  roundrobin, concurrency=8: tps=6.97, submitted=340, completed=209
  probabilistic, concurrency=1: tps=5.83, submitted=177, completed=175
  probabilistic, concurrency=2: tps=7.30, submitted=222, completed=219
  probabilistic, concurrency=4: tps=5.07, submitted=234, completed=152
  probabilistic, concurrency=8: tps=2.40, submitted=226, completed=72
  static, concurrency=1: tps=5.53, submitted=168, completed=166
  static, concurrency=2: tps=6.67, submitted=204, completed=200
  static, concurrency=4: tps=4.63, submitted=193, completed=139
  static, concurrency=8: tps=3.27, submitted=221, completed=98

Task: naive-fib
  roundrobin, concurrency=1: tps=5.87, submitted=178, completed=176
  roundrobin, concurrency=2: tps=8.13, submitt

In [78]:
import csv

os.makedirs("benchmark-results", exist_ok=True)

# Latency stats per phase
with open("benchmark-results/latency_stats.csv", "w", newline="") as f:
    w = csv.writer(f)
    w.writerow(["scheduler", "task", "rep_count", "phase",
                "n", "mean_ms", "std_ms", "median_ms", "p95_ms", "p99_ms", "iqr_ms", "ci95_low_ms", "ci95_high_ms"])
    for task_name, task_info in WORKLOADS.items():
        for rep_count in task_info.get("rep_counts", REP_COUNTS):
            for scheduler in SCHEDULERS:
                for phase, s in latency_stats.get(f"{scheduler}_{task_name}_{rep_count}", {}).items():
                    if s:
                        w.writerow([scheduler, task_name, rep_count, phase,
                                    s["n"], s["mean"], s["std"], s["median"],
                                    s["p95"], s["p99"], s["iqr"], s["ci95_low"], s["ci95_high"]])

# Energy stats
with open("benchmark-results/energy_stats.csv", "w", newline="") as f:
    w = csv.writer(f)
    w.writerow(["scheduler", "task", "rep_count", "n", "mean", "std", "median", "p95", "p99", "total"])
    for task_name, task_info in WORKLOADS.items():
        for rep_count in task_info.get("rep_counts", REP_COUNTS):
            for scheduler in SCHEDULERS:
                value = results.get(f"{scheduler}_{task_name}_{rep_count}")
                if not value:
                    continue
                vals = np.array([t["energy_consumed"] for t in value["results"] if t.get("energy_consumed") is not None], dtype=float)
                if len(vals) > 0:
                    w.writerow([scheduler, task_name, rep_count, len(vals),
                                vals.mean(), vals.std(ddof=1), np.median(vals),
                                np.percentile(vals, 95), np.percentile(vals, 99), vals.sum()])

# Throughput stats
with open("benchmark-results/throughput_stats.csv", "w", newline="") as f:
    w = csv.writer(f)
    w.writerow(["scheduler", "task", "concurrency", "throughput_tps", "submitted", "completed"])
    for task_name, task_info in WORKLOADS.items():
        if task_info.get("skip_throughput"):
            continue
        for scheduler in SCHEDULERS:
            for concurrency in THROUGHPUT_CLIENTS:
                result = throughput_benchmark_results.get(f"{scheduler}_{task_name}_{concurrency}")
                if result:
                    w.writerow([scheduler, task_name, concurrency,
                                result["throughput_tps"], result["submitted"], result["completed_before_cutoff"]])

print("Exported:")
print("  benchmark-results/latency_stats.csv")
print("  benchmark-results/energy_stats.csv")
print("  benchmark-results/throughput_stats.csv")

Exported:
  benchmark-results/latency_stats.csv
  benchmark-results/energy_stats.csv
  benchmark-results/throughput_stats.csv
